In [1]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# Pauli matrices
sigma_z = sp.Matrix([[1, 0], [0, -1]])
sigma_x = sp.Matrix([[0, 1], [1, 0]])
sigma_y = sp.Matrix([[0, -sp.I], [sp.I, 0]])
identity = sp.eye(2)

In [3]:
# Define variables
t, omega, omega_0, Omega, phi_0, epsilon, omega_m, theta_m, sig_sym_x, sig_sym_y, sig_sym_z, tau, delta = sp.symbols(
    't omega omega_0 Omega phi_0 epsilon omega_m theta_m sigma_x sigma_y sigma_z tau delta', real=True
)
hbar = sp.Symbol('hbar', real=True)  # Reduced Planck constant

In [4]:
Hamiltonian_first_frame = -delta/2 * sigma_z + Omega/2 * (sp.cos(phi_0) * sigma_x + sp.sin(phi_0) * sigma_y) + epsilon * omega_m / Omega * sp.cos(omega_m * t - theta_m) * sigma_z
Hamiltonian_first_frame



Matrix([
[-delta/2 + epsilon*omega_m*cos(omega_m*t - theta_m)/Omega,                     Omega*(-I*sin(phi_0) + cos(phi_0))/2],
[                      Omega*(I*sin(phi_0) + cos(phi_0))/2, delta/2 - epsilon*omega_m*cos(omega_m*t - theta_m)/Omega]])

## Below is attempting to solve it with phi and theta
- these probably dont matter as the quasi energies should be immune to time translations - and phases are just that
- it took far too long to solve

In [1]:
import sympy as sp

# Define variables
delta, Omega, phi_0, epsilon, omega_m, theta_m, lam = sp.symbols('delta Omega phi_0 epsilon omega_m theta_m lam')
I = sp.I  # Imaginary unit

# Pauli matrices
sigma_x = sp.Matrix([[0, 1], [1, 0]])
sigma_y = sp.Matrix([[0, -I], [I, 0]])
sigma_z = sp.Matrix([[1, 0], [0, -1]])
I2 = sp.eye(2)
omega_I = omega_m * I2

# Static part of Hamiltonian (time-averaged)
H0 = -delta/2 * sigma_z + (Omega/2) * (sp.cos(phi_0) * sigma_x + sp.sin(phi_0) * sigma_y)

# Time-dependent modulation term: Fourier components
H1 = (epsilon * omega_m / (2 * Omega)) * sp.exp(-I * theta_m) * sigma_z
H_minus1 = (epsilon * omega_m / (2 * Omega)) * sp.exp(I * theta_m) * sigma_z

# Floquet matrix blocks (6x6 for n = -1, 0, +1)
HF_full = sp.Matrix([
    [H0 - omega_I, H1, sp.zeros(2, 2)],
    [H_minus1,     H0, H1],
    [sp.zeros(2, 2), H_minus1, H0 + omega_I]
])

# Solve for quasienergies (Floquet exponents)
lam = sp.symbols('lam')
quasienergies = [sp.simplify(expr) for expr in sp.solve(sp.det(HF_full - lam * sp.eye(6)), lam)]

KeyboardInterrupt: 

## Solve numerically to confirm that theta not mattering is true




In [34]:
import numpy as np
# Parameter conventions
natural_freq = 10.02  # GHz
driving_freq = 10.0  # GHz
rabi_freq = 0.005    # GHz (5 MHz)

phi_0, epsilon_m, phase_freq, theta_m = np.pi/2, rabi_freq/4, rabi_freq, np.pi/2


def ccd_floquet(natural_freq, driving_freq, rabi_freq, phi_0, epsilon_m, phase_freq, theta_m):
    """
    Compute Floquet quasienergies for a driven two-level system.

    Parameters:
        natural_freq: float
            Natural frequency of the system (GHz).
        driving_freq: float
            Driving frequency (GHz).
        rabi_freq: float
            Rabi frequency (GHz).
        phi_0: float
            Initial phase (radians).
        epsilon_m: float
            Modulation amplitude (GHz).
        phase_freq: float
            Modulation frequency (GHz).
        theta_m: float
            Modulation phase (radians).

    Returns:
        numpy.ndarray
            Sorted real parts of the Floquet quasienergies (GHz).
    """
    # Derived parameters
    delta = natural_freq - driving_freq  # Detuning in GHz
    Omega = rabi_freq                    # Rabi frequency in GHz

    # Pauli matrices
    sigma_x = np.array([[0, 1], [1, 0]], dtype=complex)
    sigma_y = np.array([[0, -1j], [1j,  0]], dtype=complex)
    sigma_z = np.array([[1, 0], [0, -1]], dtype=complex)
    I2      = np.eye(2, dtype=complex)

    # Build H0, H1, H_{-1}
    H0       = -delta/2 * sigma_z \
            + (Omega/2) * (np.cos(phi_0)*sigma_x + np.sin(phi_0)*sigma_y)
    H1       = (epsilon_m*phase_freq/(2*Omega)) * np.exp(-1j*theta_m) * sigma_z
    H_minus1 = (epsilon_m*phase_freq/(2*Omega)) * np.exp( 1j*theta_m) * sigma_z
    omega_I  = phase_freq * I2

    # Assemble the 6×6 Floquet Hamiltonian
    HF = np.block([
        [H0 - omega_I,      H1,                   np.zeros((2,2), dtype=complex)],
        [H_minus1,          H0,                   H1],
        [np.zeros((2,2)),   H_minus1,             H0 + omega_I]
    ])

    # Diagonalize
    eigvals = np.linalg.eigvals(HF)

    # Sort and return
    return np.sort(eigvals.real)


eigvals_sorted = ccd_floquet(natural_freq, driving_freq, rabi_freq, phi_0, epsilon_m, phase_freq, theta_m)
print("Quasienergies (Floquet exponents) [in GHz]:")
for v in eigvals_sorted:
    print(v)

Quasienergies (Floquet exponents) [in GHz]:
-0.015381672284493168
-0.010310068865776584
-0.00523632864852715
0.005236328648527146
0.010310068865776565
0.015381672284493185


In [35]:
import pickle
import sympy as sp



# Load the saved symbolic data
with open('ccd_quasienergy_6.pkl', 'rb') as f_in:
    loaded_data = pickle.load(f_in)

expr = loaded_data['expression'] 
symbol_list = loaded_data['symbols']


delta_symbol, Omega_symbol, phi_0_symbol, epsilon_symbol, omega_m_symbol, theta_m_symbol, lam_symbol = sp.symbols('delta Omega phi_0 epsilon omega_m theta_m lam')
# Create symbolic variables


# Calculate value using same parameters
#note try calculating in hz if that doesnt work try rads
value = float(sp.re(expr.evalf(subs={
    delta_symbol: 0.02,
    Omega_symbol: rabi_freq,
    epsilon_symbol: epsilon_m,
    omega_m_symbol: phase_freq,
})))


In [36]:
print("Value of the expression with parameters:")
print(value-0.010310068865776565)

Value of the expression with parameters:
2.0816681711721685e-16


## Diagonalising the floquet matrix is a hell of a lot quicker 
- I have confirmed to my self that the theta doesnt matter in the expression

test floquet tools




